# Chapter 5 — Worked Example: Hydrochemistry and Nitrate Risk, Hennaya Plain

**AI for Hydrogeologists** — companion notebook

**Methodological note.** With only 19 sampling points (17 wells + 2 springs,
dry/wet seasons, 2022), a supervised classifier evaluated with a credible
train/test split (as originally outlined for Section 5.5, targeting
AUC ~0.84) is not statistically defensible. This notebook instead uses
methods that are legitimate at this sample size: unsupervised clustering,
PCA, paired non-parametric seasonal comparison, and descriptive spatial
summary — all explicitly exploratory, not predictive.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

BASE = "https://raw.githubusercontent.com/Dr-LAOUFIAbdessalam/ai-hydrogeologists/main/"
xl_path = BASE + "ch05_quality_contamination/data/raw/wet_et_dry_with_19_puits.xlsx"

import requests, io
resp = requests.get(xl_path)
raw = pd.read_excel(io.BytesIO(resp.content), sheet_name="Feuil1", header=None)
raw.head()


## Reshape the dry/wet blocks into a tidy table

In [ ]:
def extract_block(raw, start_row, n=19, season="dry"):
    cols = ["point", "x", "y", "T", "ph", "tds", "ec", "na", "ca", "mg",
            "cl", "so4", "hco3", "no3_n", "k"]
    block = raw.iloc[start_row:start_row+n, 0:15].copy()
    block.columns = cols
    block["season"] = season
    return block

dry = extract_block(raw, 3, 19, "dry")
wet = extract_block(raw, 26, 19, "wet")
df = pd.concat([dry, wet], ignore_index=True)

for c in ["x","y","T","ph","tds","ec","na","ca","mg","cl","so4","hco3","no3_n","k"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Convert NO3-N (nitrate as N) to NO3 (as used for the WHO 50 mg/L threshold)
df["no3"] = df["no3_n"] * 4.43
print(f"n = {len(df)} records ({df[\'point\'].nunique()} points x 2 seasons)")


**Units check:** the spreadsheet header says "meq/L" but the sum of major
ions matches TDS almost exactly only if the values are actually mg/L (a
labelling error in the original file, confirmed with the data owner).

In [ ]:
df["ion_sum_mgL"] = df[["na","ca","mg","cl","so4","hco3"]].sum(axis=1)
print(f"mean(sum of major ions / TDS) = {(df['ion_sum_mgL']/df['tds']).mean():.2f} "
      "(confirms mg/L, not meq/L)")


## Hydrochemical facies via k-means clustering

In [ ]:
major_ions = ["na", "ca", "mg", "cl", "so4", "hco3"]
X = StandardScaler().fit_transform(df[major_ions])

for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    print(f"k={k}: silhouette = {silhouette_score(X, labels):.3f}")

df["facies_cluster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)
print(df.groupby("facies_cluster")[major_ions + ["no3"]].mean().round(1))


## PCA on major-ion chemistry

In [ ]:
pca = PCA(n_components=3)
pcs = pca.fit_transform(X)
print("Explained variance:", pca.explained_variance_ratio_.round(3))
print("PC1 loadings:", dict(zip(major_ions, pca.components_[0].round(2))))
print("PC2 loadings:", dict(zip(major_ions, pca.components_[1].round(2))))


## Seasonal comparison (paired Wilcoxon signed-rank, n=19 pairs)

In [ ]:
dry_wide = df[df.season == "dry"].set_index("point")
wet_wide = df[df.season == "wet"].set_index("point")
common = dry_wide.index.intersection(wet_wide.index)

for param in ["na", "ca", "mg", "cl", "so4", "hco3", "no3", "ec"]:
    d = dry_wide.loc[common, param].values
    w = wet_wide.loc[common, param].values
    stat, p = stats.wilcoxon(d, w)
    sig = "*" if p < 0.05 else ""
    direction = "dry > wet" if np.median(d - w) > 0 else "wet > dry"
    print(f"{param:6s}: median diff = {np.median(d-w):+7.1f}  p={p:.3f} {sig}  ({direction})")


**Key finding:** every major ion except nitrate is higher in the dry season
(evapo-concentration effect), but **nitrate is significantly higher in the
wet season** — consistent with a "first flush" mechanism, where the first
rains leach nitrogen accumulated in agricultural soil during the dry summer
into the aquifer. This is consistent with the dominance of irrigated
cropland in the surrounding land cover (Chapter 2).

## Nitrate risk: descriptive summary against the WHO threshold

In [ ]:
who_threshold = 50
for season in ["dry", "wet"]:
    sub = df[df.season == season]
    n_exceed = (sub["no3"] > who_threshold).sum()
    print(f"{season}: mean={sub[\'no3\'].mean():.1f} mg/L, max={sub[\'no3\'].max():.1f} mg/L, "
          f"exceeding WHO 50 mg/L = {n_exceed}/{len(sub)}")


**Important, and stated without alarmism:** all 19 sampling points exceed
the WHO 50 mg/L nitrate guideline in both seasons. Given the sample covers
only one year, this should be read as a strong local signal warranting
denser, longer-term monitoring — not a precise regional contamination
statistic. The co-occurrence of the highest chloride AND nitrate values at
well `p11` in both seasons is worth flagging separately: this pattern is
more typical of localized wastewater infiltration than of diffuse
agricultural leaching alone, and merits field verification rather than a
firm causal claim from chemistry data alone.

## Addendum: Compositional Data Analysis (CLR) — does it change the conclusion?

Major-ion concentrations are **compositional** data: each sample is
implicitly constrained (charge balance), so standard Euclidean-distance
methods (PCA, k-means) on raw or naively standardized values can produce
spurious correlations from the "closure effect" — a rise in one ion
mechanically compresses the others regardless of any real relationship.

We compare the naive `StandardScaler` approach used above against a
**Centered Log-Ratio (CLR)** transform, the standard choice for
compositional hydrochemical data (over ALR, which requires an arbitrary
reference component, and ILR, whose balances are harder to interpret
physically).

In [ ]:
from sklearn.metrics import adjusted_rand_score

def clr_transform(X):
    log_X = np.log(X)
    return log_X - log_X.mean(axis=1, keepdims=True)

X_raw = df[major_ions].values
X_clr = clr_transform(X_raw)

print("--- Silhouette scores: raw StandardScaler vs CLR ---")
for k in [2, 3, 4, 5]:
    labels_raw = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    labels_clr = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_clr)
    print(f"k={k}: raw silhouette = {silhouette_score(X, labels_raw):.3f}  |  "
          f"CLR silhouette = {silhouette_score(X_clr, labels_clr):.3f}")

labels_raw_3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)
labels_clr_3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_clr)
ari = adjusted_rand_score(labels_raw_3, labels_clr_3)
print(f"\nAgreement between raw and CLR cluster assignments (k=3): "
      f"Adjusted Rand Index = {ari:.3f}  (1.0 = identical, 0.0 = random)")

pca_raw = PCA(n_components=3).fit(X)
pca_clr = PCA(n_components=3).fit(X_clr)
print(f"\nExplained variance - raw: {pca_raw.explained_variance_ratio_.round(3)}")
print(f"Explained variance - CLR: {pca_clr.explained_variance_ratio_.round(3)}")
print("\nPC1 loadings - raw:", dict(zip(major_ions, pca_raw.components_[0].round(2))))
print("PC1 loadings - CLR :", dict(zip(major_ions, pca_clr.components_[0].round(2))))


**Result: the transformation matters, not just as a technicality.** The
Adjusted Rand Index between the raw-scaled and CLR-based cluster
assignments is only ~0.13 — the two approaches group samples into
substantially different clusters, not minor variants of the same grouping.
CLR also explains more variance in PC1 (~50% vs ~31%) with a cleaner,
more physically interpretable axis dominated by chloride versus magnesium,
consistent with a real hydrochemical evolution pathway (ion exchange or
differential evaporative concentration) rather than the blended, harder to
interpret Na-Cl-SO4 axis obtained from the naive (non-compositional)
approach. The lesson: for compositional hydrochemical data, skipping the
log-ratio transform is not a neutral simplification.